# Claude Agent SDK: tool search (scale to many tools)

Every tool you give Claude has a **definition** (name, description, schema) that normally sits in the context window on **every turn**. Fifty tools can cost 10-20K tokens before the agent has done anything, and accuracy drops when 30-50+ tools compete for attention.

**Tool search** fixes both problems: Claude sees only a compact **list of tool names**, *searches* when it needs a capability, and the full schemas of the best matches (up to 5) are loaded **on demand**. You have already seen it working: that `ToolSearch` call before your custom tool in the earlier notebooks.

**What you will learn (and measure):**
1. How tool search works, and when it is on or off
2. Seeing the **`ToolSearch` call** happen
3. **Measuring the saving**: 30 tools with search on vs off
4. Configuring it: `ENABLE_TOOL_SEARCH` (`true`, `auto`, `auto:N`, `false`)
5. Writing **names and descriptions** that get found
6. Helping with the **system prompt**

## The picture

```
  WITHOUT tool search (every turn)               WITH tool search (default)

  context: [ 30 full tool schemas ........ ]     context: [ list of 30 names ]
           [ conversation ]                               [ conversation ]
                                                           │ needs "send a Slack message"
                                                           ▼
                                                  ToolSearch("slack message")  ← one extra round-trip
                                                           ▼
                                                  context += [ 3-5 matching full schemas ]
                                                  they stay loaded for later turns
                                                  (until the messages where they were found get compacted)
```

| Trade-off | Detail |
|---|---|
| **Cost** | One extra round-trip each time Claude searches |
| **Benefit** | Much smaller context on every other turn; better tool choice |
| **Rule of thumb** | Fewer than ~10 tools that fit easily: loading upfront is typically faster. Dozens or hundreds: search wins |
| **Limits** | Up to 10,000 tools in a catalog, 5 results per search by default |

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("tool-search-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. A catalog of 30 fake tools

To see the effect we need **many** tools. We generate 30 with the `@tool` decorator, across six pretend services. The handlers return canned text, so everything runs offline and costs only the model calls.

(Real catalogs come from MCP servers or your own SDK servers; tool search treats both the same.)

In [ ]:
import asyncio, json, re

from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions,
    AssistantMessage, ResultMessage, SystemMessage, ToolUseBlock,
)

CATALOG = {
    "slack":    ["send_slack_message", "list_slack_channels", "search_slack_messages", "set_slack_status", "create_slack_channel"],
    "github":   ["list_github_issues", "create_github_issue", "comment_on_pull_request", "merge_pull_request", "list_github_branches"],
    "calendar": ["list_calendar_events", "create_calendar_event", "cancel_calendar_event", "find_free_meeting_slot", "set_out_of_office"],
    "database": ["run_sql_query", "describe_table", "list_database_tables", "export_table_csv", "count_rows"],
    "weather":  ["get_current_temperature", "get_rain_forecast", "get_air_quality", "get_sunrise_sunset", "get_wind_speed"],
    "billing":  ["get_invoice", "list_unpaid_invoices", "issue_refund", "get_subscription_plan", "update_billing_address"],
}

def make_tool(service, name):
    pretty = name.replace("_", " ")
    @tool(name, f"{pretty.capitalize()} ({service} service). Use for requests about {service}.", {"detail": str})
    async def handler(args, _name=name):
        return {"content": [{"type": "text", "text": f"[{_name}] OK -> {args.get('detail', '')[:60]} (demo result)"}]}
    return handler

ALL_TOOLS = [make_tool(svc, n) for svc, names in CATALOG.items() for n in names]
big_server = create_sdk_mcp_server(name="company", tools=ALL_TOOLS)
print(len(ALL_TOOLS), "tools in the catalog")

## 2. A runner that reports what Claude did and what it cost

We print each tool call, and from the final `ResultMessage` the **token usage**. The interesting number is the **total input tokens** (`input_tokens` + cache creation + cache reads), because that is roughly "how much context did we send".

In [ ]:
async def run(prompt, label="", **opts):
    opts.setdefault("max_turns", 10)
    opts.setdefault("cwd", str(DEMO_DIR))
    calls, usage, turns, cost = [], {}, None, None
    answer = ""
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock):
                        calls.append(b.name)
                        print(f"  [call] {b.name}({json.dumps(b.input)[:110]})")
            elif isinstance(m, ResultMessage):
                usage, turns, cost = m.usage or {}, m.num_turns, m.total_cost_usd
                answer = m.result if m.subtype == "success" else f"({m.subtype})"
    except Exception as error:
        print("  (query raised:", str(error)[:200], ")")
    total_in = sum(usage.get(k, 0) or 0 for k in ("input_tokens", "cache_creation_input_tokens", "cache_read_input_tokens"))
    print(f"  ANSWER: {answer[:300]}")
    print(f"  >> {label}: turns={turns} total_input_tokens={total_in} cost=${(cost or 0):.4f} searched={'ToolSearch' in calls}")
    return {"label": label, "calls": calls, "total_in": total_in, "turns": turns, "cost": cost or 0}

## 3. Watch tool search happen (the default)

We never mention searching. Claude only knows there is a list of tool names, so it must **find** the one it needs.

In [ ]:
base = dict(mcp_servers={"company": big_server}, allowed_tools=["mcp__company__*"])

r_default = await run("What is the current temperature in Paris? Use the tools to get it.", "default (search on)", **base)

Expected shape of the output:

1. `ToolSearch` with a query (often `select:...` or keywords): Claude asks the catalog for the right tool
2. The matching tool's schema is loaded
3. The real tool call (`mcp__company__get_current_temperature`)
4. The answer

`ToolSearch` is itself a **built-in tool**. (Careful: if you set `tools=[]` to remove all built-ins, you also remove `ToolSearch`, and the search never happens. We deliberately left the built-ins alone here.)

## 4. The experiment: search on vs off

Same prompt, same 30 tools. `ENABLE_TOOL_SEARCH` is an **environment variable** read by the Claude Code process, so you set it through the `env` option (in Python, `env` is merged on top of the existing environment):

| Value | Behaviour |
|---|---|
| *(unset)* | **On.** Definitions deferred, discovered on demand (falls back to upfront on old models, non-first-party `ANTHROPIC_BASE_URL`, Azure Foundry) |
| `true` | Stay on even with a non-first-party `ANTHROPIC_BASE_URL` (proxy must forward `tool_reference` blocks) |
| `auto` | On only if deferrable definitions reach **10%** of the context window; otherwise load upfront |
| `auto:N` | Same with N% (`auto:5` activates sooner) |
| `false` | **Off.** All definitions in context on every turn |

In [ ]:
r_off = await run("What is the current temperature in Paris? Use the tools to get it.", "search OFF", env={"ENABLE_TOOL_SEARCH": "false"}, **base)

In [ ]:
print(f"{'setting':22} {'turns':>5} {'input tokens':>13} {'ToolSearch used':>16}")
for r in (r_default, r_off):
    print(f"{r['label']:22} {str(r['turns']):>5} {r['total_in']:>13} {str('ToolSearch' in r['calls']):>16}")

if r_default["total_in"] and r_off["total_in"]:
    diff = r_off["total_in"] - r_default["total_in"]
    print(f"\nOFF sent {diff:+d} input tokens compared with ON for this single question.")

**How to read this:**

- The absolute totals are large because they include Claude Code's own system prompt and built-in tool definitions. **The difference between the rows is what tool search changes.**
- With search **off**, all 30 definitions ride along on every turn. With it **on**, only names ride along, plus the schema of what was found, at the price of **one extra turn** (the search).
- In a live run of this notebook, search on used roughly **24K fewer input tokens** than off for the same question. Your numbers will differ (caching, model, wording), and **cost** can move differently from token counts because cached tokens are cheaper, so compare tokens first.
- The saving grows with tool count and schema size. Real MCP tools have long descriptions and big schemas, which is where "50 tools = 10-20K tokens" comes from.

## 5. `auto` and `auto:N`: let the SDK decide

`auto` counts the tokens of all **deferrable** definitions (MCP tools not marked `alwaysLoad`, plus on-demand built-ins; core built-ins like Bash/Read/Edit load upfront and are not counted) and compares with the model's context window:

- below the threshold: load everything upfront (no extra round-trip)
- at or above it: switch tool search on

With 30 tiny tools we are far below 10%, so `auto` should behave like "off", while `auto:0`-ish thresholds behave like "on":

In [ ]:
r_auto = await run("What is the current temperature in Paris? Use the tools to get it.", "auto (10%)", env={"ENABLE_TOOL_SEARCH": "auto"}, **base)
r_auto1 = await run("What is the current temperature in Paris? Use the tools to get it.", "auto:1 (1%)", env={"ENABLE_TOOL_SEARCH": "auto:1"}, **base)

In [ ]:
print(f"{'setting':22} {'turns':>5} {'input tokens':>13} {'searched':>9}")
for r in (r_default, r_off, r_auto, r_auto1):
    print(f"{r['label']:22} {str(r['turns']):>5} {r['total_in']:>13} {str('ToolSearch' in r['calls']):>9}")

Whether `auto:1` flips to searching depends on how big the 30 definitions really are versus 1% of the window, so the exact outcome can differ from mine. What you are checking is the **mechanism**: the `searched` column tells you which mode each setting chose.

## 6. Names and descriptions decide whether a tool gets found

The search matches the query against **tool names and descriptions**. Specific words in both help:

| Weak | Strong |
|---|---|
| `query_slack` | `search_slack_messages` |
| "Query Slack" | "Search Slack messages by keyword, channel, or date range" |
| `do_thing` | `cancel_calendar_event` |

The real search is done by the model and the API, so we cannot call it directly. But we can build a **toy keyword scorer** to see *why* wording matters. This is an illustration, not the real algorithm.

In [ ]:
# OFFLINE
def toy_search(query_text, tools, top=3):
    words = set(re.findall(r"[a-z]+", query_text.lower()))
    scored = []
    for name, desc in tools:
        haystack = set(re.findall(r"[a-z]+", (name.replace("_", " ") + " " + desc).lower()))
        scored.append((len(words & haystack), name))
    return [n for s, n in sorted(scored, reverse=True)[:top] if s > 0]

weak = [("query_slack", "Query Slack"), ("do_thing", "Does a thing"), ("get_data", "Get data")]
strong = [("search_slack_messages", "Search Slack messages by keyword, channel, or date range"),
          ("cancel_calendar_event", "Cancel a calendar event by id and notify attendees"),
          ("list_unpaid_invoices", "List invoices that are overdue or unpaid for a customer")]

for q in ["find the slack message about the outage from last week", "cancel my dentist meeting", "which customers have overdue invoices"]:
    print(f"query: {q!r}")
    print("   weak names  ->", toy_search(q, weak))
    print("   strong names->", toy_search(q, strong))

The weak catalog matches almost nothing. Real search is smarter than word overlap, but the lesson carries over: **put the nouns and verbs people will actually use into names and descriptions**.

Quick checklist for each tool:
- Name = verb + object (`search_slack_messages`, not `slack`)
- Description = what it does **and** the typical inputs/filters
- Avoid generic words (`data`, `thing`, `manager`, `handler`)
- If two tools are easily confused, say how they differ

## 7. Help the search with your system prompt

You can tell Claude **which categories** of tools exist so it knows what is worth searching for. Use the `claude_code` preset with `append`, so you add to the default prompt instead of replacing it:

In [ ]:
categories_prompt = {
    "type": "preset",
    "preset": "claude_code",
    "append": "You can search for tools to work with Slack, GitHub, calendars, databases, weather and billing.",
}

r_prompted = await run(
    "Do I have any unpaid invoices? Use the tools.",
    "with category hint",
    system_prompt=categories_prompt, **base,
)

Not a dramatic change for 30 tools, but with hundreds of tools from unfamiliar servers, a one-line category list saves Claude from guessing which words to search for.

(Full details on system prompts are in the *modifying system prompts* doc.)

## 8. Keeping a few tools always loaded

In **TypeScript**, `alwaysLoad: true` on a tool (`tool(..., extras)`) or on `createSdkMcpServer` keeps its full schema in the prompt from turn one. For MCP servers configured in `.mcp.json` / settings, the same flag exempts a server from deferral and makes the first turn wait for it.

The Python `@tool` decorator and `create_sdk_mcp_server` do not expose this flag in the version used here. In Python, if a handful of tools must always be present, the practical options are to set `ENABLE_TOOL_SEARCH=false` (everything upfront) or `auto` (upfront while the total is small).

## 9. When tool search is *not* active

| Situation | Result |
|---|---|
| Model earlier than Opus 4 / Sonnet 4 (Agent Platform: before Opus 4.5 / Sonnet 4.5 / Haiku 4.5) | Definitions loaded upfront; `ENABLE_TOOL_SEARCH` cannot override |
| `ANTHROPIC_BASE_URL` points at a non-first-party host (proxy/gateway) | Off by default; `ENABLE_TOOL_SEARCH=true` forces it on, but the proxy must forward `tool_reference` blocks or requests fail |
| Azure-hosted Microsoft Foundry | Rejected server-side; the SDK loads upfront automatically |
| `CLAUDE_CODE_DISABLE_EXPERIMENTAL_BETAS` set | Stays off (managed settings can keep it on in some versions) |
| `ToolSearch` excluded, e.g. via `disallowed_tools` | The session runs without tool search |
| HIPAA configuration | Off, cannot be turned on |

## 10. Compaction and tool search

Loaded schemas stay in context for later turns **until the SDK compacts the messages where the tool was found**. After compaction Claude simply searches again the next time it needs the tool. You will see a second `ToolSearch` call late in a long session; that is normal.

## 11. Summary

| I want to ... | Do this |
|---|---|
| Use tool search | Nothing: it is the default |
| Force it off | `env={"ENABLE_TOOL_SEARCH": "false"}` |
| Only when the catalog is big | `"auto"` or `"auto:5"` |
| Make tools easy to find | Specific verb+object names and keyword-rich descriptions |
| Orient the model | `system_prompt` preset `claude_code` + `append` listing tool categories |
| See what happened | Look for `ToolSearch` in `ToolUseBlock`s; compare `ResultMessage.usage` |
| Few tools (< ~10) | Consider turning it off; the extra round-trip is not worth it |

**Related notebooks:** `agent-sdk-custom-tools-explained` (building the tools) and `agent-sdk-mcp-servers-explained` (connecting servers).